# DATA CLEANING 

In [ ]:
"""
Prepares two datasets for model training and evaluation:

  1. LLM-annotated data  → used for train / val / test_llm splits
  2. Manually coded data → used as held-out test_manual split

Label scheme: CARDS categories 0–7 (0 = no misleading claim), stored as
one-hot encoded columns label_0 … label_7.
"""

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

## load data

In [18]:
RAW_ARTICLES  = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/raw/climate_misinfo_2023-2025.csv"
LLM_ANNOT     = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/annotated/llm_annotations.parquet"
MANUAL_ANNOT  = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/annotated/manual_annotations.xlsx"
OUT_LLM       = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/llm_data_clean.parquet"
OUT_MANUAL    = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/manual_data_clean.parquet"

LABEL_COLS    = [f"label_{i}" for i in range(8)]

## cards annotated data

In [3]:
# load data
articles = pd.read_csv(RAW_ARTICLES)
llm_annot = pd.read_parquet(LLM_ANNOT)

llm_annot.head(2)

,ArticleKey,categories,prompt_tokens,completion_tokens
0,ea4d58f9,[1_7_0],4401,41
1,ead59516,[1_7_0],4570,41


In [4]:
# rename raw subclaim column and derive top-level CARDS category from it.
# CardsSubClaim entries look like "1_3_4"; the first digit is the top-level category index.
llm_annot.rename(columns={"categories": "CardsSubClaim"}, inplace=True)

llm_annot["CardsClaim"] = llm_annot["CardsSubClaim"].apply(
    lambda arr: sorted({str(x).split("_")[0] for x in arr})
    if isinstance(arr, (list, np.ndarray)) else []
)
llm_annot.drop(columns=["completion_tokens", "prompt_tokens"], inplace=True)

In [5]:
# merge with article metadata
llm_df = articles.merge(llm_annot, on="ArticleKey")

In [6]:
# validate labels
# confirm which top-level categories are present in the data.
unique_labels = sorted({label for labels in llm_df["CardsClaim"] for label in labels})
print("Unique labels found:", unique_labels)

label_counts = Counter(label for labels in llm_df["CardsClaim"] for label in labels)
print("Label counts:", label_counts)


Unique labels found: ['0', '1', '12', '2', '3', '4', '5', '6', '7']
Label counts: Counter({'4': 14751, '3': 2871, '1': 2827, '0': 911, '7': 789, '6': 536, '2': 473, '5': 337, '12': 1})


In [8]:
# examine occurences of label 12
llm_df[llm_df["CardsClaim"].apply(lambda x: "12" in x)]

,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,Medietype,Date,Month,Year,NewText,GeneralSentiment,SentimentLabel,Ritzau-telegram,CardsSubClaim,CardsClaim
10747,ea5abac3,2024-08-18 00:00:00,"Føler du, at verden er af lave? Her er 19 gode...",Jyllands-Posten har ringet til 15 danske ekspe...,1DE GLOBALE CO2-UDLEDNINGER TOPPER MÅSKE SNART...,1319,da,LASSE MOMME - BENJAMIN JUNGDAL THORGERD JEN...,Jyllands-Posten,JYP,...,Landsdækkende dagblade,2024-08-18,8,2024,"Føler du, at verden er af lave? Her er 19 gode...",Neutral,0.0,False,"[12_0_0, 4_2_0]","[12, 4]"


In [10]:
# remove invalid label "12" from any article that contains it.
llm_df["CardsClaim"] = llm_df["CardsClaim"].apply(lambda labels: [l for l in labels if l != "12"])

In [17]:
# convert string labels to integers, then apply MultiLabelBinarizer to produce
# one-hot encoded columns. The binariser is fitted here and
# reused on the manual data to ensure identical class mappings.
llm_df["CardsClaim"] = llm_df["CardsClaim"].apply(lambda labels: [int(l) for l in labels])

mlb = MultiLabelBinarizer(classes=range(8))
llm_df[LABEL_COLS] = mlb.fit_transform(llm_df["CardsClaim"])

print(f"\nLLM data prepared: {len(llm_df)} articles")
print("Label distribution:\n", llm_df[LABEL_COLS].sum())


LLM data prepared: 17976 articles
Label distribution:
 label_0      911
label_1     2827
label_2      473
label_3     2871
label_4    14751
label_5      337
label_6      536
label_7      789
dtype: int64


In [20]:
# save clean dataframe
llm_df.to_parquet(OUT_LLM)
print(f"Saved to {OUT_LLM}")

Saved to /Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/llm_data_clean.parquet


## manually annotated data

In [21]:
# load data
manual_df = pd.read_excel(MANUAL_ANNOT)
manual_df.rename(columns={"ClaimCat": "HumanClaim", "SubClaimCat": "HumanSubClaim"}, inplace=True)

In [22]:
# parse labels
# HumanClaim is stored as a semicolon-separated string of hierarchical codes,
# e.g. "3_0_0;0_0_0". Only the first digit (top-level category) is used.
# multiple claims per article are supported. Missing values default to [0].
def parse_human_claims(cell):
    if pd.isna(cell):
        return [0]
    claims = str(cell).split(";")
    return sorted({int(c.split("_")[0]) for c in claims if c.strip()})

manual_df["HumanClaim"] = manual_df["HumanClaim"].apply(parse_human_claims)

In [23]:
# uses transform to apply the same class mapping as the
# LLM data, ensuring label_0 … label_7 are consistent across both datasets.
manual_df[LABEL_COLS] = mlb.transform(manual_df["HumanClaim"])

print(f"\nManual data prepared: {len(manual_df)} articles")
print("Label distribution:\n", manual_df[LABEL_COLS].mean().round(3))


Manual data prepared: 500 articles
Label distribution:
 label_0    0.902
label_1    0.002
label_2    0.002
label_3    0.002
label_4    0.082
label_5    0.008
label_6    0.012
label_7    0.010
dtype: float64


In [24]:
# save clean dataframe
manual_df.to_parquet(OUT_MANUAL)
print(f"Saved to {OUT_MANUAL}")

Saved to /Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/manual_data_clean.parquet


In [ ]:
# 